# 课后练习解答（02.08_npu_migration）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** GPU 代码迁移到 NPU 时，设备选择最稳妥的写法是？
A. device = "npu:0" if torch.npu.is_available() else "cpu"
B. device = "cuda:0" if torch.cuda.is_available() else "cpu"
C. device = "npu:0" 无条件
D. device = "cpu"

**解答：** A

**解析：** 优先 NPU 并保留 CPU 回退，才能在不同环境间复用。


### 问题2（单选题）

**题目：** NPU 计时为什么必须在 forward 后调用 synchronize？
A. NPU kernel 异步执行，需要等待完成再取时间戳
B. 释放显存
C. 防止过拟合
D. 触发反向传播

**解答：** A

**解析：** 计时语句不会等待 NPU kernel，不同步会严重低估真实耗时。


### 问题3（多选题）

**题目：** GPU 到 NPU 迁移需要检查？
A. import torch_npu
B. torch.cuda.* 替换为 torch.npu.*
C. device 字符串由 cuda 改为 npu
D. 数据/模型/标签全部 to(device)

**解答：** ABCD

**解析：** 四类改动缺一不可，尤其是统一 device 与同步 API。


### 问题4（多选题）

**题目：** NPU 性能对比实验应固定？
A. batch_size 与输入 shape
B. warmup 与 repeats
C. dtype
D. 同步时机

**解答：** ABCD

**解析：** 任一变量不同都会让结果失去可比性。


### 问题5（判断题）

**题目：** 纯 NPU 环境导入 torch_npu 后，torch.cuda.is_available() 通常仍为 False，设备选择不能依赖 CUDA。

**解答：** 对

**解析：** NPU 环境不提供 CUDA runtime，必须用 torch.npu 接口判断。


### 问题6（判断题）

**题目：** NPU 不支持 bf16，混合精度训练必须使用 fp16 且无需梯度缩放。

**解答：** 错

**解析：** NPU 支持 bf16；低精度训练仍需配合 GradScaler 或等价机制防止梯度下溢。


### 问题7（填空题）

**题目：** 查看 NPU 算力利用率与内存占用使用命令 ____；Python 侧分析算子耗时建议使用 ____。

**解答：** npu-smi info；torch.npu.profiler（或 torch_npu.profiler）


### 问题8（填空题）

**题目：** 推理显存不足时，最直接有效的调整是减小 ____ 或 ____。

**解答：** batch_size；输入分辨率/序列长度


### 问题9（简答题）

**题目：** 为什么 NPU 首次推理耗时通常显著高于后续推理？

**解答：** 首次推理包含算子构图、内核编译、缓存预热和内存分配等一次性开销；后续推理复用已编译 kernel 与缓存，耗时回归稳态。


### 问题10（简答题）

**题目：** 如何确认模型和张量确实位于 NPU 而不是 CPU？

**解答：** 打印 model.device、next(model.parameters()).device 与 input.device，确认均为 npu:0；再执行一次 npu 上的前向与回拷，并检查 torch.npu.is_available()。


### 问题11（代码设计题）

**题目：** 编写 benchmark_inference(model, x, warmup, repeats)，要求包含同步、计时循环、返回平均耗时与吞吐。

**解答：** ```python
def benchmark_inference(model, x, warmup=10, repeats=50):
    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            model(x)
        torch.npu.synchronize()
        start = time.perf_counter()
        for _ in range(repeats):
            model(x)
        torch.npu.synchronize()
        total = time.perf_counter() - start
    ms_per_iter = total / repeats * 1000
    throughput = x.size(0) * repeats / total
    return ms_per_iter, throughput
```


### 问题12（单选题）

**题目：** NPU 利用率低而 CPU 利用率高，通常意味着？
A. 数据加载/预处理成为瓶颈
B. 算子计算密集
C. 显存不足
D. 模型太小

**解答：** A

**解析：** CPU 忙于取数与预处理、NPU 等待数据，说明供数速度跟不上计算。


### 问题13（多选题）

**题目：** 混合精度迁移中需要注意？
A. 权重/激活使用 bf16 或 fp16
B. 损失缩放与梯度裁剪配合
C. BN 等敏感层可保持 fp32
D. 所有 API 都必须改 dtype

**解答：** ABC

**解析：** 混合精度按层选择精度，不需要全模型强制改 dtype。


### 问题14（判断题）

**题目：** model、input、label 必须全部迁移到同一 device 才能正常前向与 loss 计算。

**解答：** 对

**解析：** 跨 device 张量运算会报错或触发隐式拷贝，必须统一设备。


### 问题15（简答题）

**题目：** 如何使用 profiler 定位 NPU 空闲等待数据加载的问题？请给出观察指标与改进方案。

**解答：** 运行 torch.npu.profiler 采集 step 时间线，观察算子间隙与 DataLoader gap；若设备空闲时间占比高、CPU 预处理热点明显，则调大 num_workers、优化 transform、增大共享内存或使用异步预取。
